# Costruzione del dataset pulito

## Setup

In [ ]:
!apt-get -qq install -y unrar
!pip -q install requests scipy pandas

In [ ]:
import requests, re, os, glob, subprocess
import numpy as np
from scipy.io import loadmat
from scipy.signal import resample

URL = 'https://groups.uni-paderborn.de/kat/BearingDataCenter/'
fs = 64000
signal_name = 'phase_current_1'

## Scarichiamo i cuscinetti delle 3 classi (escluso il combinato KB)

In [ ]:
codes = re.findall(r'([A-Z]+\d+)\.rar', requests.get(URL).text)
codes = sorted(set(c for c in codes if not c.startswith('KB')))

# per un primo collaudo puoi usare pochi cuscinetti:  codes = codes[:3]
for code in codes:
    if not glob.glob(os.path.join(code, '**', '*.mat'), recursive=True):
        with open(code + '.rar', 'wb') as f:
            f.write(requests.get(URL + code + '.rar').content)
        os.makedirs(code, exist_ok=True)
        subprocess.run(['unrar', 'x', '-o+', code + '.rar', code + '/'])
len(codes)

## Troviamo L: il giro piu' lungo (velocita' minima nei dati)

In [ ]:
min_speed = np.inf
for code in codes:
    for mat_file in glob.glob(os.path.join(code, '**', '*.mat'), recursive=True):
        m = loadmat(mat_file, simplify_cells=True)
        key = [k for k in m if not k.startswith('__')][0]
        speed = next(y['Data'] for y in m[key]['Y'] if y['Name'] == 'speed')
        min_speed = min(min_speed, speed.min())

L = round(fs / (min_speed / 60))
print('velocita minima:', int(min_speed), 'rpm  ->  L =', L)

## Le trasformazioni (segnale -> frame)

In [ ]:
def samples_per_revolution(rpm):
    return round(fs / (rpm / 60))

def segment_by_revolution(signal, samples_per_rev):
    signal = np.asarray(signal).ravel()
    n = signal.size // samples_per_rev
    return signal[:n * samples_per_rev].reshape(n, samples_per_rev)

def resample_frames(frames, target_length):
    return resample(frames, target_length, axis=1)

def standardize(frames):
    frames = np.asarray(frames, dtype=np.float64)
    mean = frames.mean(axis=1, keepdims=True)
    std = frames.std(axis=1, keepdims=True)
    std[std == 0] = 1.0
    return (frames - mean) / std

## Etichette: classe e natura dal codice

In [ ]:
real = {'KA04','KA15','KA16','KA22','KA30','KI04','KI14','KI16','KI17','KI18','KI21'}
artificial = {'KA01','KA03','KA05','KA06','KA07','KA08','KA09','KI01','KI03','KI05','KI07','KI08'}
class_by_prefix = {'K': 0, 'KA': 1, 'KI': 2}

def prefix(code):
    return code.rstrip('0123456789')

def nature(code):
    if prefix(code) == 'K':
        return 'sano'
    return 'reale' if code in real else 'artificiale'

## Costruiamo il dataset

In [ ]:
X, y, meta = [], [], []
for code in codes:
    for mat_file in sorted(glob.glob(os.path.join(code, '**', '*.mat'), recursive=True)):
        parts = os.path.basename(mat_file).split('_')
        condition = '_'.join(parts[:3])
        m = loadmat(mat_file, simplify_cells=True)
        key = [k for k in m if not k.startswith('__')][0]
        current = next(s['Data'] for s in m[key]['Y'] if s['Name'] == signal_name)
        speed = next(s['Data'] for s in m[key]['Y'] if s['Name'] == 'speed')
        rpm = speed.mean()
        frames = segment_by_revolution(current, samples_per_revolution(rpm))
        frames = standardize(resample_frames(frames, L))
        X.append(frames.astype(np.float32))
        cls = class_by_prefix[prefix(code)]
        y.extend([cls] * len(frames))
        meta.extend([(code, nature(code), condition, rpm)] * len(frames))

X = np.vstack(X)
y = np.array(y)
print('X:', X.shape, '| y:', y.shape)

## Salviamo su Google Drive

In [ ]:
import pandas as pd, json
from google.colab import drive
drive.mount('/content/drive')

out = '/content/drive/MyDrive/bearings_detection'
os.makedirs(out, exist_ok=True)
np.savez(os.path.join(out, 'dataset.npz'), X=X, y=y)
pd.DataFrame(meta, columns=['bearing','nature','condition','rpm']).to_csv(os.path.join(out, 'anagrafica.csv'), index=False)
json.dump({'fs': fs, 'signal': signal_name, 'L': L}, open(os.path.join(out, 'config.json'), 'w'))